In [1]:
import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import openpyxl

def melon(keyword, max_page=30):
    url = "https://www.melon.com/index.htm"
    options = Options()
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()
    driver.get(url)
    wait = WebDriverWait(driver, 10)
    time.sleep(1)

    search_box = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "#top_search"))
    )

    search_box.clear()
    search_box.send_keys(keyword)
    search_box.send_keys(Keys.ENTER)

    time.sleep(3)

    artist = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="divCollection"]/ul/li[2]/a/span')
        )
    )
    artist.click()

    singer = wait.until(
        EC.element_to_be_clickable(
            (By.CSS_SELECTOR, '#pageList > div > ul > li:nth-child(1) > div > div > dl > dt > a')
        )
    )
    singer.click()
    song = wait.until(
        EC.element_to_be_clickable(
            (By.CSS_SELECTOR, '#conts > div.wrap_tab_atist.type9 > ul > li:nth-child(3) > a > span')
        )
    )
    song.click()
    time.sleep(1)
    very_end = wait.until(
        EC.element_to_be_clickable(
            (By.CSS_SELECTOR, '#pageObjNavgation > div > a.btn_last')
        )
    )

    very_end.click()

    

    data = []
    while True:
        time.sleep(3)
        current_url = driver.current_url
        current_url = current_url[-4:]
        page = int(re.sub(r'[^0-9]', '' ,current_url))

        wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "div.tb_list.d_song_list.songTypeOne tbody tr")
            )
        )
        melon_parsing(driver, data)
        if (page - 50 <= 0):
            break
        driver.execute_script(
            f"pageObj.sendPage('{page - 50}');")
    driver.quit()
    df = pd.DataFrame(data)
    df["번호"] = df["번호"].astype('int')
    df = df.sort_values(by = ["번호"]).set_index("번호")

    df.to_excel('melon.xlsx')
    return df
    

def melon_parsing(driver, data):
    html = driver.page_source

    soup = BeautifulSoup(html, "html.parser")
    song_lists = soup.select('div.tb_list.d_song_list.songTypeOne tr')
    
    for song_list in song_lists:
        title_tag = song_list.select_one("a.fc_gray")
        number_tag = song_list.select_one("td.no div")
        like_tag = song_list.select_one("span.cnt")

        title = title_tag.get_text(strip=True) if title_tag else ""
        number = number_tag.get_text(strip=True) if number_tag else ""
        like = like_tag.get_text(strip=True).replace("총건수", "") if like_tag else ""
        if not number or title == "":
            continue

        data.append({
            "번호": number,
            "제목": title,
            "좋아요": like
        })
    return data
melon("악뮤")

,제목,좋아요
번호,,
1,소문의 낙원,"83,734"
2,봄 색깔,"8,265"
3,벌레를 내고,"9,060"
4,"기쁨, 슬픔, 아름다운 마음","85,717"
5,햇빛 bless you,"9,776"
...,...,...
97,사랑은 은하수 다방에서,"27,818"
98,Crescendo (크레셴도),"68,877"
99,라면인건가,"34,144"


In [2]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

def elegant_brothers():
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    driver.get("https://techblog.woowahan.com/")
    time.sleep(2)
    data = []

    # 다음 페이지로 넘어가기
    
    for i in range(1, 60, 1):
        try:
            if i == 2:
                page = wait.until(
                    EC.element_to_be_clickable(
                        (By.CSS_SELECTOR, 'body > div.content.vuejs > div.content-wrap > div.page-main > div.post-main > div.pagination > div > a:nth-child(2)')
                    )
                )
                page.click()

            if 6 > i > 2:
                page = wait.until(
                    EC.element_to_be_clickable(
                        (By.CSS_SELECTOR, f'body > div.content.vuejs > div.content-wrap > div.page-main > div.post-main > div.pagination > div > a:nth-child({i+1})')
                    )
                )
                    
                page.click()
            if i >= 6:
                page = wait.until(
                    EC.element_to_be_clickable(
                        (By.CSS_SELECTOR, f'body > div.content.vuejs > div.content-wrap > div.page-main > div.post-main > div.pagination > div > a:nth-child(6)')
                    )
                    
                )
                    
                page.click()
                time.sleep(1)
            if i >= 51:
                page = wait.until(
                    EC.element_to_be_clickable(
                        (By.CSS_SELECTOR, f'body > div.content.vuejs > div.content-wrap > div.page-main > div.post-main > div.pagination > div > a.page.larger')
                    )
                )
                page.click()
                time.sleep(1)
        except TimeoutException:
            break

        # 각 포스트 들어간 다음 되돌아오기
        for j in range(11, 20):
            try:
                post = wait.until(
                    EC.element_to_be_clickable(
                        (By.CSS_SELECTOR, f'body > div.content.vuejs > div.content-wrap > div.page-main > div.post-main > div.post-list > div:nth-child({j}) > a > h2')
                    )
                )
            except TimeoutException:
                break
            post.click()
            time.sleep(1)
            data = ele_sele(wait, data)
            
            driver.back()
            time.sleep(0.5)
        print(f"{i}번째 크롤링을 완료했습니다.")


    df = pd.DataFrame(data)
    driver.quit()

    return df

    

def ele_sele(wait, data):
    
    title = wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, 'body > div.content.vuejs > div.content-wrap.content-single > div.post-content > div.post-header > h1')
        )
    )
    data.append({
        "제목": title.text
    })
    return data

elegant_brothers()

1번째 크롤링을 완료했습니다.
2번째 크롤링을 완료했습니다.
3번째 크롤링을 완료했습니다.
4번째 크롤링을 완료했습니다.
5번째 크롤링을 완료했습니다.


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=149.0.7827.199); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff78ad03fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff78ad04000+14980]
	chromedriver!(No symbol) [0x7ff78a84793d]
	chromedriver!(No symbol) [0x7ff78a832d82]
	chromedriver!(No symbol) [0x7ff78a8590cc]
	chromedriver!(No symbol) [0x7ff78a8d1aa0]
	chromedriver!(No symbol) [0x7ff78a8eec82]
	chromedriver!(No symbol) [0x7ff78a89401c]
	chromedriver!(No symbol) [0x7ff78a894f43]
	chromedriver!GetHandleVerifier [0x7ff78b2e7591+5f7f11]
	chromedriver!GetHandleVerifier [0x7ff78b2e1902+5f2282]
	chromedriver!GetHandleVerifier [0x7ff78b307115+617a95]
	chromedriver!GetHandleVerifier [0x7ff78ad21dce+3274e]
	chromedri

In [50]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# HTML 페이지 요청
url = "https://www.naver.com/"

response = requests.get(url)

# HTML 파싱
soup = BeautifulSoup(response.text, "html.parser")

# img 태그 찾기
img_tag = soup.find("img")

# 이미지 주소 가져오기
img_src = img_tag.get("src")
print(img_src)

# 상대경로 → 절대경로 변환
img_url = urljoin(url, img_src)

print("이미지 주소:", img_url)

# images 폴더 생성
os.makedirs("images", exist_ok=True)

# 이미지 다운로드
img_response = requests.get(img_url)

# 파일 이름 추출
# 'http://127.0.0.1:5500/류지.jpeg'
# ['http:', '', '127.0.0.1:5500', '류지.jpeg']
file_name = img_url.split("/")[-1]
# file_name = '류지.jpeg'

# 저장 경로
save_path = os.path.join("images", file_name)
# save_path = 'images\\류지.jpeg'

AttributeError: 'NoneType' object has no attribute 'get'

In [ ]:
import pandas as pd

# 첫 번째 크롤링 데이터
data1 = [
    {"제목": "파이썬 기초", "조회수": 120},
    {"제목": "판다스 사용법", "조회수": 95},
]

# 두 번째 크롤링 데이터
data2 = [
    {"제목": "셀레니움 기초", "조회수": 180},
    {"제목": "웹 크롤링", "조회수": 210},
]

# 세 번째 크롤링 데이터
data3 = [
    {"제목": "머신러닝 입문", "조회수": 300},
    {"제목": "딥러닝 기초", "조회수": 250},
]

# 각각 데이터프레임으로 변환
df1 = pd.DataFrame(data1)
df2 = pd.DataFrame(data2)
df3 = pd.DataFrame(data3)

# 하나의 엑셀 파일에 시트를 나눠서 저장
with pd.ExcelWriter("크롤링결과.xlsx", engine="openpyxl") as writer:
    df1.to_excel(writer, sheet_name="시트1", index=False)
    df2.to_excel(writer, sheet_name="시트2", index=False)
    df3.to_excel(writer, sheet_name="시트3", index=False)

print("엑셀 저장 완료!")